# Modelling Table
This notebook combines the cleaned rental data, property-level spatial features and SA2-level external features into a single modelling-ready dataset.

## Load Data

In [1]:
import pandas as pd

domain_df = pd.read_parquet("../data/analysis_data/")
sa2_df = pd.read_parquet("../data/curated/sa2_features.parquet")
spatial_df = pd.read_parquet("../data/curated/property_geo_features.parquet")

In [2]:
print(domain_df.columns.tolist())
print(spatial_df.columns.tolist())
print(sa2_df.columns.tolist())

['listing_id', 'suburb', 'postcode', 'lat', 'lon', 'weekly_rent', 'bond', 'available_date', 'date_listed', 'days_listed', 'bedrooms', 'bathrooms', 'carspaces', 'scraped_date', 'photo_count', 'video_count', 'floorplans_count', 'virtual_tour', 'property_type', 'primary_type', 'secondary_type', 'available_date_clean', 'available_weekday', 'bond_imputed', 'carspaces_imputed']
['property_id', 'sa2_code_2021', 'sa2_name_2021', 'in_victoria', 'straight_km_to_cbd', 'route_km_to_cbd', 'route_min_to_cbd', 'straight_km_to_nearest_station', 'est_route_km_to_nearest_station', 'nearest_station_name', 'n_stations_within_1km', 'n_stations_within_2km', 'straight_km_to_nearest_school', 'nearest_school_name', 'n_schools_within_1km', 'n_schools_within_2km', 'straight_km_to_nearest_park', 'straight_km_to_nearest_named_park', 'nearest_named_park', 'n_parks_within_500m', 'n_parks_within_1km', 'straight_km_to_nearest_mall', 'nearest_mall_name']
['sa2_code_2021', 'sa2_name_2021', 'sa3_name', 'sa4_name', 'gcc_n

## Validate Join Keys

In [3]:
print(domain_df["listing_id"].dtype)
print(spatial_df["property_id"].dtype)

print(spatial_df["sa2_code_2021"].dtype)
print(sa2_df["sa2_code_2021"].dtype)

str
str
str
str


In [4]:
print("Domain listing_id duplicates:",
        domain_df["listing_id"].duplicated().sum())

print("Spatial property_id duplicates:",
        spatial_df["property_id"].duplicated().sum())

print("SA2 code duplicates:",
        sa2_df["sa2_code_2021"].duplicated().sum())

Domain listing_id duplicates: 0
Spatial property_id duplicates: 0
SA2 code duplicates: 0


## Merge Data

In [5]:
# Align join keys
domain_df["listing_id"] = domain_df["listing_id"].astype(str)
spatial_df["property_id"] = spatial_df["property_id"].astype(str)

spatial_df["sa2_code_2021"] = spatial_df["sa2_code_2021"].astype(str)
sa2_df["sa2_code_2021"] = sa2_df["sa2_code_2021"].astype(str)

# Domain + property-level spatial features
master_df = domain_df.merge(
    spatial_df,
    left_on="listing_id",
    right_on="property_id",
    how="left",
    validate="one_to_one",
    indicator="spatial_merge_status"
)

print("After spatial merge:", master_df.shape)
print(master_df["spatial_merge_status"].value_counts())

assert (master_df["spatial_merge_status"] == "both").all()

After spatial merge: (11945, 49)
spatial_merge_status
both          11945
left_only         0
right_only        0
Name: count, dtype: int64


In [6]:
master_df = master_df.merge(
    sa2_df,
    on="sa2_code_2021",
    how="left",
    validate="many_to_one",
    suffixes=("", "_sa2"),
    indicator="sa2_merge_status"
)

print("Final master shape:", master_df.shape)
print(master_df["sa2_merge_status"].value_counts())

assert (master_df["sa2_merge_status"] == "both").all()

Final master shape: (11945, 87)
sa2_merge_status
both          11945
left_only         0
right_only        0
Name: count, dtype: int64


## Quality Checks

In [7]:
print("Rows:", len(master_df))
print("Unique listings:", master_df["listing_id"].nunique())

print("Missing SA2 match:",
        master_df["sa2_code_2021"].isna().sum())

print("Missing population_2026 values:",
        master_df["population_2026"].isna().sum())

Rows: 11945
Unique listings: 11945
Missing SA2 match: 0
Missing population_2026 values: 0


In [8]:
print("Master shape:", master_df.shape)

missing = (master_df.isna().sum().sort_values(ascending=False))

print(missing[missing > 0])

Master shape: (11945, 87)
nearest_mall_name    801
dtype: int64


In [9]:
# Check ranges of key modelling variables
cols = [
    "weekly_rent",
    "bedrooms",
    "bathrooms",
    "carspaces",
    "route_km_to_cbd",
    "route_min_to_cbd",
    "straight_km_to_nearest_station",
    "income_median",
    "population_growth_5y"
]

print(master_df[cols].describe())

        weekly_rent      bedrooms     bathrooms     carspaces  \
count  11945.000000  11945.000000  11945.000000  11945.000000   
mean     626.757974      2.734701      1.588949      1.613646   
std      288.566871      1.076770      0.630210      0.805310   
min       50.000000      0.000000      1.000000      0.000000   
25%      490.000000      2.000000      1.000000      1.000000   
50%      560.000000      3.000000      2.000000      2.000000   
75%      685.000000      4.000000      2.000000      2.000000   
max    10000.000000     11.000000     12.000000     15.000000   

       route_km_to_cbd  route_min_to_cbd  straight_km_to_nearest_station  \
count     11945.000000      11945.000000                    11945.000000   
mean         45.076115         44.370626                        3.608832   
std          63.588531         44.684501                       11.926957   
min           0.871840          1.873500                        0.018700   
25%          10.963520         19.

In [10]:
print(master_df["is_modelling_ready"].value_counts(dropna=False))
print(master_df["is_analysable"].value_counts(dropna=False))

is_modelling_ready
True     11943
False        2
Name: count, dtype: int64
is_analysable
True     11943
False        2
Name: count, dtype: int64


In [11]:
excluded_df = master_df[~master_df["is_modelling_ready"]][["listing_id", "suburb", "sa2_code_2021", "sa2_name_2021", "is_analysable", "is_modelling_ready", "is_residential", "income_reliable"]]

excluded_df

,listing_id,suburb,sa2_code_2021,sa2_name_2021,is_analysable,is_modelling_ready,is_residential,income_reliable
4530,17743279,FLEMINGTON,206041120,Flemington Racecourse,False,False,True,False
4544,17702625,FLEMINGTON,206041120,Flemington Racecourse,False,False,True,False


In [12]:
assert master_df["listing_id"].is_unique
assert master_df["sa2_code_2021"].notna().all()
assert master_df["population_2026"].notna().all()
assert (master_df["listing_id"] == master_df["property_id"]).all()
assert (master_df["sa2_name_2021"] == master_df["sa2_name_2021_sa2"]).all()

In [13]:
master_df = master_df.drop(columns=["property_id", "sa2_name_2021_sa2", "spatial_merge_status", "sa2_merge_status"])

## Output

In [14]:
# Full merged table
master_df.to_parquet("../data/curated/master_modelling_table_full.parquet",
                        index=False)

# Modelling-ready sample only
modelling_df = master_df[master_df["is_modelling_ready"]].copy()

modelling_df.to_parquet("../data/curated/master_modelling_table.parquet",
                        index=False)

modelling_df.to_csv("../data/curated/master_modelling_table.csv", index=False)

In [15]:
print("Full:", master_df.shape)
print("Modelling-ready:", modelling_df.shape)

Full: (11945, 83)
Modelling-ready: (11943, 83)
